<a href="https://colab.research.google.com/github/tobiasllop/Tesis/blob/main/Rankings_predeterminados.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import polars as pl
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

try:
    import kaleido
    print("Kaleido installed. PDF/image export enabled.")
    kaleido_installed = True
except ImportError:
    print("Kaleido not installed. Install with `pip install kaleido` for PDF/image export.")
    kaleido_installed = False

print("1. Cargando datos y seleccionando columnas necesarias...")
df_bcra = pl.scan_parquet("/content/drive/MyDrive/Tesis2026/deudores_enero_2026_clasificados.parquet").select([
    "cod_entidad", "segmento", "deuda_total"
])

entidades_maestro = pl.scan_csv("/content/drive/MyDrive/Tesis2026/EntidadesFinancierasBCRA.csv").select([
    "cod_entidad", "nombre_entidad"
])

print("2. Estandarizando códigos y cruzando bases...")
df_bcra = df_bcra.with_columns(pl.col("cod_entidad").cast(pl.Utf8).str.strip_chars().str.zfill(5))
entidades_maestro = entidades_maestro.with_columns(pl.col("cod_entidad").cast(pl.Utf8).str.strip_chars().str.zfill(5))

df_joined = df_bcra.join(entidades_maestro, on="cod_entidad", how="left")

print("3. Calculando métricas agrupadas por segmento y entidad...")
# Agrupamos y calculamos tanto la cantidad de deudas como la suma del capital
df_agg = df_joined.group_by(["segmento", "nombre_entidad"]).agg([
    pl.len().alias("cantidad_deudas"),
    pl.col("deuda_total").sum().alias("monto_prestado")
]).collect(streaming=True)

# Pasamos a pandas solo el resumen agregado para graficar (es muy liviano)
df_plot = df_agg.to_pandas().dropna(subset=["segmento"])

# Filtramos el segmento "Sin Deuda" como solicitado
df_plot = df_plot[df_plot['segmento'] != 'Sin Deuda']

print("4. Generando los 6 gráficos comparativos...")
segmentos_unicos = df_plot['segmento'].unique()

# Iteramos sobre cada segmento para armar los gráficos
for seg in segmentos_unicos:
    # Filtramos los datos del segmento actual
    df_seg = df_plot[df_plot['segmento'] == seg]

    # Obtenemos el Top 10 por Cantidad de Deudas
    top_cantidad = df_seg.nlargest(10, 'cantidad_deudas').sort_values('cantidad_deudas', ascending=True)

    # Obtenemos el Top 10 por Monto Prestado
    top_monto = df_seg.nlargest(10, 'monto_prestado').sort_values('monto_prestado', ascending=True)

    # Creamos una figura con 2 subplots (1 fila, 2 columnas)
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=(
            f"Top 10 por CANTIDAD de Deudas",
            f"Top 10 por MONTO Prestado ($)"
        ),
        horizontal_spacing=0.15
    )

    # Gráfico 1: Barras Horizontales para Cantidad
    fig.add_trace(
        go.Bar(
            y=top_cantidad['nombre_entidad'],
            x=top_cantidad['cantidad_deudas'],
            orientation='h',
            marker=dict(color='#2b7a78'),
            name='Cantidad',
            text=top_cantidad['cantidad_deudas'].apply(lambda x: f"{x:,.0f}"),
            textposition='outside'
        ),
        row=1, col=1
    )

    # Gráfico 2: Barras Horizontales para Monto
    fig.add_trace(
        go.Bar(
            y=top_monto['nombre_entidad'],
            x=top_monto['monto_prestado'],
            orientation='h',
            marker=dict(color='#3aafa9'),
            name='Monto ($)',
            text=top_monto['monto_prestado'].apply(lambda x: f"${x:,.0f}"),
            textposition='outside'
        ),
        row=1, col=2
    )

    # Ajustes estéticos para que se vea profesional
    fig.update_layout(
        title_text=f"📊 Análisis Competitivo: Deudas de Segmento {str(seg).upper()}",
        title_font_size=20,
        height=500,
        width=1200,
        showlegend=False,
        plot_bgcolor='white',
        margin=dict(l=250, r=50, t=80, b=50) # Margen izquierdo amplio para los nombres de bancos
    )

    fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='LightGray')

    fig.show()

    # Exportar los gráficos a PNG y PDF
    if kaleido_installed:
        file_name_prefix = f"Analisis_Competitivo_Segmento_{str(seg).upper().replace(' ', '_')}"
        pio.write_image(fig, f"{file_name_prefix}.png")
        pio.write_image(fig, f"{file_name_prefix}.pdf")
        print(f"Exported {file_name_prefix}.png and {file_name_prefix}.pdf")


Kaleido installed. PDF/image export enabled.
1. Cargando datos y seleccionando columnas necesarias...
2. Estandarizando códigos y cruzando bases...
3. Calculando métricas agrupadas por segmento y entidad...


/tmp/ipykernel_17539/3202305958.py:34: DeprecationWarning: the `streaming` parameter was deprecated in 1.25.0; use `engine` instead.
  ]).collect(streaming=True)


4. Generando los 6 gráficos comparativos...


RuntimeError: 

Kaleido requires Google Chrome to be installed.

Either download and install Chrome yourself following Google's instructions for your operating system,
or install it from your terminal by running:

    $ plotly_get_chrome



In [ ]:
# Install Google Chrome, which Kaleido requires for static image export
# This command is often necessary in Colab environments.
!sudo apt-get update
!sudo apt-get install -y google-chrome-stable

# Alternatively, Kaleido provides its own way to get Chrome:
# !plotly_get_chrome # This might require a separate installation of `plotly_get_chrome` utility

print("Google Chrome installation initiated. Please run the plotting cell (XfF0mrNyyPEy) again after this cell completes.")

Hit:1 https://cli.github.com/packages stable InRelease
Get:2 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [95.6 kB]
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:9 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,947 kB]
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:11 http://security.ubuntu.com/ubuntu jammy-security/multiverse amd64 Packages [77.8 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,295 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [

In [ ]:
!pip install --upgrade plotly

In [ ]:
!pip install kaleido